# Automated Report Generation: Excel · HTML · PDF 


## Sample Data (simulate pipeline output)

In [11]:
import pandas as pd
import numpy as np
from datetime import datetime

np.random.seed(42)

dates = pd.date_range("2026-01-01", periods=30, freq="D")
df = pd.DataFrame({
    "Date": dates,
    "Region": np.random.choice(["North", "South", "East", "West"], 30),
    "Sales": np.random.randint(800, 4500, 30),
    "Profit": np.random.uniform(120, 980, 30).round(2),
    "Orders": np.random.randint(15, 120, 30)
})

# Add some KPIs
summary = df.groupby("Region").agg({
    "Sales": "sum",
    "Profit": "sum",
    "Orders": "sum"
}).round(2).reset_index()

total_sales = df["Sales"].sum()
top_region = summary.loc[summary["Sales"].idxmax(), "Region"]

print("Sample data ready. Total sales:", total_sales, " Top region:", top_region)

Sample data ready. Total sales: 84302  Top region: East


## 3. Excel Report – formatted + chart

In [12]:
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import PatternFill, Font, Alignment

excel_path = f"monthly_report_{datetime.now().strftime('%Y%m')}.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    # Summary sheet
    summary.to_excel(writer, sheet_name="Summary", index=False, startrow=2)
    ws = writer.sheets["Summary"]
    ws["A1"] = "Monthly Sales Report"
    ws["A1"].font = Font(bold=True, size=16)

    # Raw data sheet
    df.to_excel(writer, sheet_name="Raw Data", index=False)

    # Conditional formatting (openpyxl style)
    green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
    for row in ws.iter_rows(min_row=4, min_col=2, max_col=5):
        for cell in row:
            if cell.column_letter in ["C", "D"] and cell.value > 2000:
                cell.fill = green_fill

    # Add bar chart
    chart = BarChart()
    chart.title = "Sales by Region"
    chart.y_axis.title = "Total Sales"
    chart.x_axis.title = "Region"
    data = Reference(ws, min_col=2, min_row=3, max_row=ws.max_row)
    cats = Reference(ws, min_col=1, min_row=4, max_row=ws.max_row)
    chart.add_data(data, titles_from_data=True)
    chart.set_categories(cats)
    ws.add_chart(chart, "G2")

print(f"Excel report saved → {excel_path}")

Excel report saved → monthly_report_202603.xlsx


## 4. Styled HTML Report

In [13]:
def highlight_high_sales(val):
    color = "#d4edda" if val > 3000 else ""
    return f"background-color: {color}"

styled = summary.style\
    .format({"Sales": "{:,}", "Profit": "{:.2f}", "Orders": "{:,}"})\
    .applymap(highlight_high_sales, subset=["Sales"])\
    .set_caption(f"Monthly Report – {datetime.now().strftime('%B %Y')}")\
    .set_table_styles([
        {"selector": "caption", "props": "caption-side: top; font-size: 1.5em; font-weight: bold;"},
        {"selector": "th", "props": "background-color: #f2f2f2; text-align: center;"},
    ])

html_path = f"monthly_report_{datetime.now().strftime('%Y%m')}.html"
styled.to_html(html_path, index=False)

# Optional: wrap in full HTML with Jinja if you want header/footer/logo
print(f"Styled HTML saved → {html_path}")
print("Open in browser or attach to email.")

Styled HTML saved → monthly_report_202603.html
Open in browser or attach to email.


/tmp/ipykernel_55284/3070880062.py:7: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(highlight_high_sales, subset=["Sales"])\


## PDF Report (WeasyPrint)

In [14]:
from weasyprint import HTML

html_content = f"""
<html>
<head>
<style>
    body {{ font-family: Arial, sans-serif; margin: 40px; }}
    h1 {{ color: #2c3e50; }}
    table {{ border-collapse: collapse; width: 100%; margin: 20px 0; }}
    th, td {{ border: 1px solid #ddd; padding: 12px; text-align: right; }}
    th {{ background-color: #f2f2f2; }}
    .highlight {{ background-color: #d4edda; }}
</style>
</head>
<body>
<h1>Monthly Sales Report – {datetime.now().strftime('%B %Y')}</h1>
<p><strong>Total Sales:</strong> {total_sales:,} | <strong>Top Region:</strong> {top_region}</p>
<table>
<tr><th>Region</th><th>Sales</th><th>Profit</th><th>Orders</th></tr>
"""

for _, row in summary.iterrows():
    highlight = ' class="highlight"' if row["Sales"] > 10000 else ""
    html_content += f"<tr{highlight}><td>{row['Region']}</td><td>{row['Sales']:,}</td><td>{row['Profit']:.2f}</td><td>{row['Orders']:,}</td></tr>\n"

html_content += "</table></body></html>"

pdf_path = f"monthly_report_{datetime.now().strftime('%Y%m')}.pdf"
HTML(string=html_content).write_pdf(pdf_path)

print(f"Professional PDF saved → {pdf_path}")

Professional PDF saved → monthly_report_202603.pdf


## Alternative: Simple PDF with fpdf2

In [17]:
from fpdf import FPDF

class PDF(FPDF):
    def header(self):
        self.set_font("Arial", "B", 15)
        self.cell(0, 10, "Monthly Sales Report", 0, 1, "C")
        self.ln(10)

pdf = PDF()
pdf.add_page()
pdf.set_font("Arial", size=12)

pdf.cell(0, 10, f"Total Sales: {total_sales:,} : Top: {top_region}", ln=True)
pdf.ln(10)

# Table header
pdf.set_font("Arial", "B", 12)
pdf.cell(40, 10, "Region", border=1)
pdf.cell(40, 10, "Sales", border=1)
pdf.cell(40, 10, "Profit", border=1)
pdf.cell(40, 10, "Orders", border=1)
pdf.ln()

# Data rows
pdf.set_font("Arial", size=12)
for _, row in summary.iterrows():
    pdf.cell(40, 10, row["Region"], border=1)
    pdf.cell(40, 10, f"{row['Sales']:,}", border=1)
    pdf.cell(40, 10, f"{row['Profit']:.2f}", border=1)
    pdf.cell(40, 10, f"{row['Orders']:,}", border=1)
    pdf.ln()

simple_pdf = "simple_monthly_report.pdf"
pdf.output(simple_pdf)
print(f"Simple PDF (pure Python) → {simple_pdf}")

Simple PDF (pure Python) → simple_monthly_report.pdf
